# In V2 Self-Attentation (Single Head)

In [136]:
import torch

import torch.nn as nn

# Data

In [137]:
vocab = {
    "I" : 0,
    "Love" : 1,
    "AI" : 2
}

tokens = torch.tensor([0, 1, 2]) # or torch.tensor(vocab.values())

# Embeddings

In [138]:
embedding_dim = 8

embedding = nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = embedding_dim
)

In [139]:
x  = embedding(tokens)

print(x.shape)
print(x)

torch.Size([3, 8])
tensor([[-0.9463, -0.9239,  0.3127, -0.6225, -1.0961, -0.1065,  0.5703, -0.1879],
        [-0.4052, -0.8336, -0.0517,  0.3191, -0.6827,  0.0404,  0.1341,  0.3615],
        [ 0.1523, -0.9453, -1.5585, -0.7048,  1.0953, -0.2183, -1.5687, -0.4673]],
       grad_fn=<EmbeddingBackward0>)


# Create Q, K, V Projections

- insted of finding linear for one at a time
- find for all

In [140]:
d_model = 8

# Linear Objects
wq = nn.Linear(in_features = 8, out_features = 8)
wk = nn.Linear(in_features = 8, out_features = 8)
wv = nn.Linear(in_features = 8, out_features = 8)

Q = wq(x)
K = wk(x)
V = wv(x)

print(Q.shape)
print(K.shape)
print(V.shape)

torch.Size([3, 8])
torch.Size([3, 8])
torch.Size([3, 8])


In [141]:
print(Q)
print(K)
print(V)

tensor([[ 3.8023e-01, -3.8532e-02,  5.6861e-01,  2.2327e-01,  1.9892e-01,
         -9.0205e-01, -1.2504e-02, -2.5874e-01],
        [ 4.9685e-02,  2.1433e-02, -7.1427e-04, -3.6305e-02,  4.8989e-01,
         -4.2221e-01, -1.3963e-01, -3.2547e-04],
        [ 7.8956e-01, -6.7589e-01,  1.1283e-01,  1.4337e-01, -6.3432e-01,
          1.0361e-01,  2.2887e-01, -1.5427e-01]], grad_fn=<AddmmBackward0>)
tensor([[-0.3426, -0.3886, -0.5494,  0.5746,  0.1000, -0.1959, -0.5828,  0.5767],
        [-0.3212,  0.0300, -0.5590,  0.0408, -0.1085, -0.4201, -0.0147,  0.1775],
        [-0.6712,  0.7877,  0.3260, -0.5376, -0.5326,  0.2051,  0.3561, -0.3586]],
       grad_fn=<AddmmBackward0>)
tensor([[-0.5274,  0.4309,  0.4460,  0.0239,  0.1772,  0.5164,  1.1599, -0.4490],
        [-0.5276,  0.4213, -0.0770,  0.0427,  0.2984,  0.1100,  0.5972, -0.1753],
        [-0.3669,  0.3550, -1.4751, -0.0662, -0.1349,  0.7985, -0.3915, -0.1788]],
       grad_fn=<AddmmBackward0>)


# what are Q K V?

### Query (Q)
- what am i looking for?

### Key (K)
- what information do i contain?

### Value (V)
- what information should i send?


# Compute Attentation Scores

In [142]:
# Q -> 3, 8
# K -> 3, 8 -> K.T -> 8, 3


# Q
# [[ 0.5072,  0.1735, -0.7746,  0.7097, -0.1132,  0.0097, -0.8604,  0.0785],   ->  I
# [-0.3272, -0.1659, -0.3873,  0.5849,  0.3650, -0.2599, -0.7006, -0.2199],    -> Love
# [ 0.5718,  0.7499,  0.2985, -0.6165, -0.1292, -0.4832,  0.0266, -0.5100]]    -> AI

#            X (dot product)

# K
#     I        Love      AI
# [[-0.7450, -0.1356, -0.0721],
# [-0.5689, -0.0183,  0.8617],
# [-0.1136, -0.1002,  0.2329],
# [ 0.2167,  0.4746, -0.7126],
# [ 0.3051, -0.1228,  0.1295],
# [-0.3004,  0.2029,  0.0043],
# [-0.1476, -0.4845,  0.2799],
# [-0.5763,  0.0497,  0.3345]]

#     =

#    II       ILove    IAI
# [-0.1905,  0.7791, -0.8024],

#  LoveI    LoveLove   LoveAI
# [ 0.9285,  0.5947, -0.8500],

#    AII     AILove    AIAI
# [-0.6244, -0.5342,  0.9318]]

In [143]:
scores = Q @ K.T # Dot product

print(scores.shape)
scores

torch.Size([3, 3])


tensor([[-0.2447, -0.1205, -0.4229],
        [ 0.1671,  0.1098, -0.3943],
        [-0.2936, -0.3366, -0.6067]], grad_fn=<MmBackward0>)

# Softmax

- Converts scores into probability

In [144]:
weights = torch.softmax(scores, dim = - 1)

print(weights)

tensor([[0.3368, 0.3814, 0.2818],
        [0.3977, 0.3755, 0.2268],
        [0.3719, 0.3562, 0.2719]], grad_fn=<SoftmaxBackward0>)


# Dont forgot each time i run i will get different result

23% -> I

63% -> Love

12% -> AI

# Weighted sum

In [145]:
weight = weights @ V

weight.shape

torch.Size([3, 8])

# Example:

#### Consider:

- bank of river

vs

- bank account

#### The embedding for:

- bank -> starts the same.

#### But after attention:

- bank (river context)

- becomes different from

- bank (finance context)

because it looked at neighboring tokens.

This is one of the reasons Transformers became so powerful.

# Problem in this implementation 

In [146]:
embedding_dim = 128

embedding = nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = embedding_dim
)

In [147]:
tokens = torch.tensor([0, 1, 2])
embedded_tokens = embedding(tokens)

print(embedded_tokens.shape)

d_model = 128

# Linear Objects
wq = nn.Linear(in_features = 128, out_features = 128)
wk = nn.Linear(in_features = 128, out_features = 128)
wv = nn.Linear(in_features = 128, out_features = 128)

print(wq)
print(wk)
print(wv)

Q = wq(embedded_tokens)
K = wk(embedded_tokens)
V = wv(embedded_tokens)

scores = Q @ K.T # Dot product

print(scores.shape)

print(scores.max())
print(scores.min())

torch.Size([3, 128])
Linear(in_features=128, out_features=128, bias=True)
Linear(in_features=128, out_features=128, bias=True)
Linear(in_features=128, out_features=128, bias=True)
torch.Size([3, 3])
tensor(2.0722, grad_fn=<MaxBackward1>)
tensor(-14.5439, grad_fn=<MinBackward1>)


# What is the problem

- when i use dot product for 3, 8  and 8, 3 its ok because q1k1+q2k2....+q8k8 = new number
- when i use 3, 128 and 128, 3 is q1k1 + .... + q128k128

##### the number will be big 

*3 x 8 matrix*

- scores = [2, 1, 0]

- Softmax: [0.66, 0.24, 0.10]

*3 x 128 matrix*

- scores = [20, 10, 0]

- [0.99995, 0.00005, 0.00000]

In [ ]:
scores

tensor([[-14.5439,  -8.8141,   1.1872],
        [ -2.1837,  -6.7650,  -4.0128],
        [ -3.7571,   2.0722,  -5.5178]], grad_fn=<MmBackward0>)

: 

In [148]:
scores[1, 2]

tensor(-4.0128, grad_fn=<SelectBackward0>)